In [55]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json
from string import Template

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import utils


In [56]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [57]:
# agenda items dataframe

agenda_items_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized.json")
    with open(input_filepath, 'r', encoding='utf-8') as f:
        agenda_items = json.load(f)
    for item in agenda_items:
        my_row = {
            "date": date,
            "year": year,
            "item_no": item['item_no'],
            "sub_item_no": item['sub_item_no'],
            "is_consent_calendar": item['is_consent_calendar'],
            "agenda_text": item['text']
        }
        for col in ['title', 'address', 'council_district', 'council_member', 'last_day_to_act', 'referenced_laws', 'is_appeal', 'summary_of_appeal', 'summary']:
            if col=='referenced_laws':
                my_row[col] = item['ai_summary'].get(col, [])
            else:
                my_row[col] = item['ai_summary'].get(col, "")
        my_row['agenda_summary'] = my_row.pop('summary')
        my_row['is_casenum'] = utils.is_casenum(my_row['title'])
        agenda_items_df.append(my_row)

agenda_items_df = pd.DataFrame(agenda_items_df)
agenda_items_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "agenda_items.parquet"), index=False)
    

In [58]:
# minutes items dataframe

minutes_items_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-cleaned-summarized-minutes.json")
    with open(input_filepath, 'r', encoding='utf-8') as f:
        minutes_items = json.load(f)
    for item in minutes_items:
        if not item['ai_minutes_summary']:
            continue
        my_row = {
            "date": date,
            "year": year,
            "item_no": item['item_no'],
            "sub_item_no": item['sub_item_no'],
        }
        for col in ['summary', 'item_order_changed', 'taken_off_consent_calendar', 'multiple_votes', 'mover', 'seconder', 'ayes', 'noes', 'abstentions', 'absences', 'recusals', 'vote_result', 'appeal_result', 'project_result']:
            if col in ['ayes', 'noes', 'abstentions', 'absences', 'recusals']:
                my_row[col] = item['ai_minutes_summary'].get(col, [])
            else:
                my_row[col] = item['ai_minutes_summary'].get(col, "")
        my_row['minutes_summary'] = my_row.pop('summary')
        minutes_items_df.append(my_row)

minutes_items_df = pd.DataFrame(minutes_items_df)
minutes_items_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "minutes_items.parquet"), index=False)   


In [59]:
# supplemental docs dataframe

supplemental_docs_df = []
for _, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs-summarized.json")
    docs_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "supplemental-docs.pkl")
    docs_df = pd.read_pickle(docs_filepath)
    with open(input_filepath, 'r', encoding='utf-8') as f:
        supplemental_docs = json.load(f)
    for doc in supplemental_docs:
        my_row = {
            "date": date,
            "year": year,
            "doc_id": doc["doc_id"],
            "text": docs_df.loc[(docs_df["date"]==date) & (docs_df["doc_id"]==doc["doc_id"]), "content"].values[0]
        }
        for col in ['summary', 'referenced_agenda_items', 'document_type', 'author_type', 'support_or_oppose']:
            my_row[col] = doc[col]
        supplemental_docs_df.append(my_row)

supplemental_docs_df = pd.DataFrame(supplemental_docs_df)
supplemental_docs_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "supplemental_docs.parquet"), index=False)
            


In [60]:
# Main analysis dataframe

df = minutes_items_df.merge(
    agenda_items_df, 
    on=['date', 'year', 'item_no', 'sub_item_no'],
    how='inner'
).reset_index(drop=True)

# some cleaning

df['is_appeal'] = (df['is_appeal']=="yes")
df['item_order_changed'] = (df['item_order_changed']=="YES")
df['taken_off_consent_calendar'] = (df['taken_off_consent_calendar']=="YES")
df['multiple_votes'] = (df['multiple_votes']=="YES")

# name standardization

with open(os.path.join(LOCAL_PATH, "intermediate_data/cpc/names-standardized.json"), 'r', encoding='utf-8') as f:
    names_mapping = json.load(f)

df['mover'] = df['mover'].map(names_mapping)
df['seconder'] = df['seconder'].map(names_mapping)
df['ayes'] = df['ayes'].apply(lambda x: [names_mapping[name] for name in x])
df['noes'] = df['noes'].apply(lambda x: [names_mapping[name] for name in x])
df['abstentions'] = df['abstentions'].apply(lambda x: [names_mapping[name] for name in x])
df['absences'] = df['absences'].apply(lambda x: [names_mapping[name] for name in x])
df['recusals'] = df['recusals'].apply(lambda x: [names_mapping[name] for name in x])

# num ayes and noes
df['num_ayes'] = df['ayes'].apply(len)
df['num_noes'] = df['noes'].apply(len)
df['unanimous'] = (df['num_ayes']>0) & (df['num_noes']==0)

# count support vs opposition
df['n_docs'] = 0
df['n_support'] = 0
df['n_oppose'] = 0
df['n_docs_individual'] = 0
df['n_support_individual'] = 0
df['n_oppose_individual'] = 0
for _, row in supplemental_docs_df.iterrows():
    date = row['date']
    year = row['year']
    for item in row['referenced_agenda_items']:
        mask = (df['date']==date) & (df['year']==year) & (df['item_no']==item) 
        df.loc[mask, 'n_docs'] += 1
        if row['author_type']=="INDIVIDUAL":
            df.loc[mask, 'n_docs_individual'] += 1
    for k, v in row['support_or_oppose'].items():
        mask = (df['date']==date) & (df['year']==year) & (df['item_no']==k)
        if v in ["DEFINITELY SUPPORT", "SOMEWHAT SUPPORT"]:
            df.loc[mask, 'n_support'] += 1
            if row['author_type']=="INDIVIDUAL":
                df.loc[mask, 'n_support_individual'] += 1
        elif v in ["DEFINITELY OPPOSE", "SOMEWHAT OPPOSE"]:
            df.loc[mask, 'n_oppose'] += 1
            if row['author_type']=="INDIVIDUAL":
                df.loc[mask, 'n_oppose_individual'] += 1

df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet"), index=False)
